# NLP Pipeline + RAG System

**Techniques covered:**
- Natural Language Processing (NLP) — intent classification, entity extraction, text preprocessing
- Retrieval-Augmented Generation (RAG) — menu retrieval via semantic search

Both are required AI techniques for the Advanced AI module.

In [ ]:
import json
import re
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
import faiss

DATA_DIR = Path('../data')

with open(DATA_DIR / 'synthetic_dataset.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

with open(DATA_DIR / 'menu_data.json', 'r', encoding='utf-8') as f:
    menu = json.load(f)

print(f'Dataset: {len(dataset)} conversations')
print(f'Menu items: {len(menu["items"])}')

## Part A — NLP Pipeline

### A1. Text Preprocessing for Sinhala

In [ ]:
class SinhalaPreprocessor:
    SHORTHAND = {
        'eka':'1','deka':'2','thuna':'3','hathara':'4',
        'ekath':'1','withrak':'only 1',
        'onen':'want','oenei':'want',
        'denna':'give','denn':'give',
        'karanna':'do','karann':'do',
        'tiyenavada':'available?','tiyenawa':'available',
        'machang':'','yep':'yes','nah':'no',
        'plz':'please','pls':'please',
    }

    def normalize_unicode(self, text):
        return re.sub(r'[\u200b\u200c\u200d\ufeff]', '', text).strip()

    def lowercase_roman(self, text):
        parts = re.split(r'([\u0D80-\u0DFF]+)', text)
        return ''.join(p if re.match(r'[\u0D80-\u0DFF]+', p) else p.lower() for p in parts)

    def normalize_shorthand(self, text):
        tokens = text.split()
        return ' '.join(self.SHORTHAND.get(t, t) for t in tokens)

    def remove_noise(self, text):
        text = re.sub(r'[^\w\s\u0D80-\u0DFF.,?!]', ' ', text)
        return re.sub(r'\s+', ' ', text).strip()

    def detect_language(self, text):
        si = len(re.findall(r'[\u0D80-\u0DFF]', text))
        en = len(re.findall(r'[a-zA-Z]', text))
        total = si + en or 1
        r = si / total
        return 'sinhala' if r > 0.65 else 'english' if r < 0.25 else 'mixed'

    def preprocess(self, text):
        text = self.normalize_unicode(text)
        text = self.lowercase_roman(text)
        text = self.remove_noise(text)
        text = self.normalize_shorthand(text)
        return text


pp = SinhalaPreprocessor()

examples = [
    ('චිකන් කොට්ටු දෙකක් ඕන', 'sinhala formal order'),
    ('machang 3 hot mutton kottu!!!', 'slang English order'),
    ('Hi, fried rice order කරන්න ඕන', 'code-switched'),
    ('yes confirm', 'confirmation'),
]

print('Text Preprocessing Examples:')
print('-' * 70)
for text, label in examples:
    proc = pp.preprocess(text)
    lang = pp.detect_language(text)
    print(f'[{label}]')
    print(f'  Raw:       {text}')
    print(f'  Processed: {proc}')
    print(f'  Language:  {lang}')
    print()

### A2. Intent Classification

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Flatten dataset into (text, intent) pairs
rows = []
for conv in dataset:
    user_turns = [t for t in conv['conversation'] if t['role'] == 'user']
    intents = conv['labels']['intent_sequence']
    for turn, intent in zip(user_turns, intents):
        rows.append({'text': turn['content'], 'intent': intent, 'lang': conv['language']})

df = pd.DataFrame(rows)
df['processed'] = df['text'].apply(pp.preprocess)

print(f'Total utterances: {len(df)}')
print(f'Unique intents: {df["intent"].nunique()}')
print(f'\nDistribution:')
print(df['intent'].value_counts().to_string())

# Train/test split
intent_counts = df['intent'].value_counts()
valid = intent_counts[intent_counts >= 2].index
df_f = df[df['intent'].isin(valid)]

X_tr, X_te, y_tr, y_te = train_test_split(
    df_f['processed'].values, df_f['intent'].values,
    test_size=0.2, random_state=42, stratify=df_f['intent'].values
)

# TF-IDF + Logistic Regression (character n-grams — works well for Sinhala)
clf = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4),
                              max_features=8000, sublinear_tf=True)),
    ('lr', LogisticRegression(max_iter=1000, C=1.5, class_weight='balanced'))
])
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

macro_f1 = f1_score(y_te, y_pred, average='macro', zero_division=0)
print(f'\nIntent Classifier — Macro F1: {macro_f1:.3f}')
print(classification_report(y_te, y_pred, zero_division=0))

### A3. Named Entity Recognition (Entity Extraction)

In [ ]:
class EntityExtractor:
    def __init__(self, items):
        self._map = {}
        for it in items:
            self._map[it['name'].lower()] = it['name']
            self._map[it['sinhala']] = it['name']
            self._map[it['name'].lower().replace(' ','')] = it['name']

        self._spice = re.compile(r'\b(very hot|extra hot|hot|medium|mild)\b', re.I)
        self._qty = re.compile(r'\b(\d+|one|two|three|four|eka|deka|thuna)\b', re.I)
        self._addr = re.compile(
            r'[A-Za-z ]+,\s*[A-Za-z ]+\d+|\d+[/\\]\d+[A-Za-z]?|'
            r'[A-Za-z]+ (?:road|street|lane|rd|st)\s*\d*', re.I)
        self._num = {'one':1,'two':2,'three':3,'four':4,'eka':1,'deka':2,'thuna':3}

    def extract(self, text):
        food = [c for k,c in self._map.items() if k in text.lower() or k in text]
        food = list(dict.fromkeys(food))

        qtys = [self._num.get(m.lower(), int(m)) if not m.isdigit() else int(m)
                for m in self._qty.findall(text)]

        spice = (m := self._spice.search(text)) and m.group(1).lower()
        addr = (m := self._addr.search(text)) and m.group(0).strip()

        return {'food_items': food, 'quantities': qtys,
                'spice_level': spice or None, 'address': addr or None}


extractor = EntityExtractor(menu['items'])

ner_tests = [
    'chicken kottu 2 saha watalappan 1 medium karanna',
    'mutton kottu 3 hot. Dehiwala, Beach Road 15',
    'egg kottu deka mild please Kottawa main street 12',
    'වටලප්පන් දෙකයි, චිකන් කොට්ටු එකයි ඕනේ',
]

print('Entity Extraction:')
print('=' * 60)
for t in ner_tests:
    ents = extractor.extract(t)
    print(f'Input: {t}')
    for k, v in ents.items():
        if v:
            print(f'  {k}: {v}')
    print()

## Part B — RAG (Retrieval-Augmented Generation)

The RAG system retrieves relevant menu items given a customer query, and injects them into the LLM context. This ensures the model always has accurate, up-to-date pricing without hallucinating.

### B1. Build the FAISS index

In [ ]:
encoder = SentenceTransformer('all-MiniLM-L6-v2')

# Build corpus
items = menu['items']
corpus = []
for it in items:
    parts = [it['name'], it.get('sinhala',''), it.get('category',''), it.get('description','')]
    if it.get('addons'):
        parts += it['addons']
    corpus.append(' '.join(p for p in parts if p))

print('Encoding menu items...')
embeddings = encoder.encode(corpus, show_progress_bar=True, convert_to_numpy=True)

# FAISS index
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings.astype(np.float32))

print(f'\nFAISS index built: {index.ntotal} vectors, dim={embeddings.shape[1]}')

### B2. Query the RAG system

In [ ]:
def rag_search(query: str, top_k: int = 3):
    vec = encoder.encode([query], convert_to_numpy=True).astype(np.float32)
    dists, idxs = index.search(vec, top_k)
    results = []
    for d, i in zip(dists[0], idxs[0]):
        if i < len(items):
            r = dict(items[i])
            r['_distance'] = float(d)
            results.append(r)
    return results


rag_queries = [
    'chicken kottu',
    'vegetarian options',
    'dessert something sweet',
    'කොට්ටු',
    'something to drink cola',
]

print('RAG Retrieval Results:')
print('=' * 60)
for q in rag_queries:
    results = rag_search(q)
    print(f'Query: "{q}"')
    for r in results:
        avail = '✓' if r.get('available', True) else '✗'
        print(f'  {avail} {r["name"]} ({r["sinhala"]}) Rs.{r["price_lkr"]}  [dist={r["_distance"]:.2f}]')
    print()

### B3. RAG Context Injection Example

In [ ]:
def build_rag_context(query: str) -> str:
    """Build the context string to inject into the LLM prompt."""
    hits = rag_search(query, top_k=4)
    lines = ['[RETRIEVED MENU ITEMS]']
    for h in hits:
        avail = 'Available' if h.get('available', True) else 'NOT available'
        lines.append(f'- {h["name"]} ({h["sinhala"]}) Rs.{h["price_lkr"]} — {avail}')
        if h.get('addons'):
            lines.append(f'  Add-ons: {", ".join(h["addons"])}')
    return '\n'.join(lines)


demo_query = 'I want chicken kottu with extra egg'
print('Demo: RAG context for query:', demo_query)
print('-' * 50)
ctx = build_rag_context(demo_query)
print(ctx)
print()
print('[This context is injected into the system prompt before the LLM generates a response]')

### B4. RAG Retrieval Accuracy

In [ ]:
# Evaluate: does the top-1 retrieved item match the gold label?
gold_queries = [
    ('chicken kottu please', 'Chicken Kottu'),
    ('mutton kottu', 'Mutton Kottu'),
    ('egg fried rice', 'Egg Fried Rice'),
    ('watalappan dessert', 'Watalappan'),
    ('vegetable noodles', 'Vegetable Noodles'),
    ('coca cola', 'Coca Cola (330ml)'),
    ('samosa short eat', 'Samosa (2 pcs)'),
    ('family meal deal offer', 'Family Meal Deal'),
]

correct = 0
for q, gold in gold_queries:
    hits = rag_search(q, top_k=1)
    predicted = hits[0]['name'] if hits else 'None'
    match = gold.lower() in predicted.lower() or predicted.lower() in gold.lower()
    correct += match
    icon = '✓' if match else '✗'
    print(f'{icon} Query: "{q}" → {predicted} (gold: {gold})')

accuracy = correct / len(gold_queries)
print(f'\nRAG Top-1 Accuracy: {accuracy:.1%} ({correct}/{len(gold_queries)})')

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['RAG Top-1\nAccuracy'], [accuracy], color='#4C72B0', width=0.3)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Accuracy')
ax.set_title('RAG Menu Retrieval Performance')
ax.text(0, accuracy + 0.02, f'{accuracy:.1%}', ha='center', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('../data/rag_accuracy.png', dpi=150)
plt.show()